# Tugas Akhir 04 (TA-04): Prediksi Data Sekuensial Menggunakan LSTM
**Studi Kasus:** Peramalan Harga Penutupan Emas (Gold Futures - Ticker: `GC=F`) Berbasis Deret Waktu (Time Series)

### Alur Kerja Proyek:
1. **Load Dataset & Preprocessing**: Pembersihan data, parsing datetime, dan pengecekan missing value.
2. **Exploratory Data Analysis (EDA)**: Visualisasi grafik tren historis harga penutupan emas.
3. **Data Scaling**: Normalisasi nilai harga menggunakan `MinMaxScaler(0, 1)`.
4. **Windowing / Sequence Creation**: Transformasi data sekuens dengan timestep 30 hari ke belakang untuk memprediksi hari berikutnya.
5. **Split Data & Tensor Reshape**: Pembagian 80% data latih dan 20% data uji, lalu di-reshape ke Tensor 3D `(samples, timesteps, features)`.
6. **Pembangunan Arsitektur Model Sequential LSTM**: Membangun multi-layer LSTM dengan Dropout dan Dense layer.
7. **Pelatihan Model & Visualisasi Loss**: Melatih model dengan optimizer Adam dan loss MSE serta memantau validation loss.
8. **Evaluasi Performa**: Melakukan *inverse transform*, menghitung metrik regresi (RMSE & MAE), dan memvisualisasikan perbandingan nilai aktual vs prediksi.
9. **Penyimpanan Artefak**: Menyimpan model ke format `.h5` dan objek scaler menggunakan `joblib` untuk dideploy pada aplikasi web Flask.

--- 
## 1. Import Library & Memuat Dataset

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# 1. Membaca dataset time-series
csv_path = 'dataset_timeseries.csv'
df = pd.read_csv(csv_path)

# Deteksi kolom tanggal secara fleksibel
date_cols = [col for col in df.columns if 'date' in str(col).lower()]
date_col = date_cols[0] if date_cols else df.columns[0]

# Parsing kolom Date ke tipe datetime dan set sebagai index
df[date_col] = pd.to_datetime(df[date_col])
df.set_index(date_col, inplace=True)
df.sort_index(inplace=True)

# Memastikan kolom 'Close' bertipe numerik dan membersihkan missing value
df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
df.dropna(subset=['Close'], inplace=True)

print(f"[INFO] Kolom indeks tanggal terdeteksi: '{date_col}'")
print(f"[INFO] Total baris data perdagangan: {len(df)}")
print(f"[INFO] Rentang data: {df.index.min().strftime('%Y-%m-%d')} s/d {df.index.max().strftime('%Y-%m-%d')}")
df.head()

--- 
## 2. Eksplorasi Data (EDA) & Visualisasi Tren Historis

In [ ]:
# Visualisasi pergerakan harga penutupan emas secara historis
plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Close'], color='#d4af37', linewidth=2.0, label='Harga Penutupan (Close)')
plt.title('Grafik Tren Historis Harga Emas (Gold Futures - GC=F)', fontsize=14, fontweight='bold')
plt.xlabel('Tanggal', fontsize=11)
plt.ylabel('Harga (USD/troy oz)', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

--- 
## 3. Scaling Data (Normalisasi Menggunakan MinMaxScaler)

In [ ]:
# Ambil nilai kolom target 'Close' sebagai array 2D
close_prices = df[['Close']].values

# Normalisasi ke rentang [0, 1]
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(close_prices)

print(f"Bentuk data ter-skala: {scaled_data.shape}")
print(f"Nilai Minimum: {scaled_data.min():.4f} | Nilai Maksimum: {scaled_data.max():.4f}")

--- 
## 4. Pembuatan Sekuens (Windowing)
Menggunakan $30$ hari data perdagangan sebelumnya untuk memprediksi harga pada hari ke-$31$.

In [ ]:
def create_sequences(data, time_step=30):
    """
    Mengubah array time series 1D menjadi matriks sekuens input (X) dan target (y).
    """
    X, y = [], []
    for i in range(len(data) - time_step):
        X.append(data[i:(i + time_step), 0])
        y.append(data[i + time_step, 0])
    return np.array(X), np.array(y)

TIME_STEP = 30
X, y = create_sequences(scaled_data, time_step=TIME_STEP)

print(f"Bentuk matriks X (Input) : {X.shape}")
print(f"Bentuk matriks y (Target): {y.shape}")

--- 
## 5. Split Data Train-Test & Reshape Tensor 3D
Data dibagi menjadi 80% data latih (*training*) dan 20% data uji (*testing*) secara berurutan waktu (tanpa acak/shuffle).

In [ ]:
train_size = int(len(X) * 0.8)

X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Reshape matriks input X menjadi Tensor 3D: (samples, timesteps, features)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"X_train shape (3D): {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape  (3D): {X_test.shape}  | y_test shape : {y_test.shape}")

--- 
## 6. Pengembangan Arsitektur Model Sequential LSTM

In [ ]:
# Membangun arsitektur Sequential LSTM
model = Sequential([
    LSTM(units=50, return_sequences=True, input_shape=(TIME_STEP, 1)),
    Dropout(0.2),
    LSTM(units=50, return_sequences=False),
    Dropout(0.2),
    Dense(units=1)
])

# Compile model dengan Optimizer Adam dan Mean Squared Error (MSE)
model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

--- 
## 7. Pelatihan Model (Training) & Monitoring Loss

In [ ]:
# Melatih model dengan pemantauan performa pada data validasi
history = model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

# Visualisasi grafik penurunan Loss (MSE)
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Training Loss (MSE)', color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss (MSE)', color='orange')
plt.title('Grafik Konvergensi Loss Pelatihan vs Validasi')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

--- 
## 8. Evaluasi Kinerja Model: Inverse Transform, RMSE, MAE, & Plot Prediksi

In [ ]:
# Melakukan prediksi pada data uji
predictions = model.predict(X_test)

# Inverse transform hasil prediksi dan target ke skala mata uang asli (USD)
predictions_actual = scaler.inverse_transform(predictions)
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

# Menghitung metrik regresi: RMSE dan MAE
rmse = np.sqrt(mean_squared_error(y_test_actual, predictions_actual))
mae = mean_absolute_error(y_test_actual, predictions_actual)

print("=" * 50)
print("HASIL EVALUASI METRIK REGRESI:")
print(f"  - Root Mean Squared Error (RMSE) : ${rmse:.2f}")
print(f"  - Mean Absolute Error (MAE)     : ${mae:.2f}")
print("=" * 50)

# Plot perbandingan harga aktual vs hasil prediksi
plt.figure(figsize=(12, 5))
plt.plot(y_test_actual, color='blue', label='Harga Aktual (Ground Truth)')
plt.plot(predictions_actual, color='red', linestyle='--', label='Hasil Prediksi Model LSTM')
plt.title('Evaluasi Model: Perbandingan Harga Emas Aktual vs Prediksi')
plt.xlabel('Indeks Waktu (Data Uji)')
plt.ylabel('Harga Emas (USD/troy oz)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

--- 
## 9. Penyimpanan Model dan Scaler untuk Integrasi Web Flask

In [ ]:
# Membuat folder 'model' jika belum ada
os.makedirs('model', exist_ok=True)

# Menyimpan model Keras (.h5) dan Scaler (.pkl)
model.save('model/model_lstm.h5')
joblib.dump(scaler, 'model/scaler.pkl')

print("[SUKSES] Seluruh artefak siap digunakan oleh web server Flask app.py:")
print(" 1. model/model_lstm.h5")
print(" 2. model/scaler.pkl")